# Sequence Prediction - STRICT protocol

Every stage runs with `--strict`, which switches the pipeline from the
permissive script default to the evaluation population Screen2Vec itself uses in
`modeltester.py`:

| | permissive (script default) | `--strict` (this notebook) |
|---|---|---|
| traces kept | >= 3 screens | >= `n_predictors + 1` (= 5) |
| context per prediction | 1..4 screens (truncated near trace start) | exactly 4 |
| candidate pool | every embedding in the directory | only screens of eligible traces |
| top-k% thresholds | `ceil()` | `int()` truncation, as in `modeltester.py:141` |
| training negatives | any screen | only screens of eligible traces |

Because every training window is full under `--strict`, the zero-padding branch
in the loss never fires, so training and evaluation see the same context length.
That mismatch is present in the permissive pipeline.

## Both evaluations live in this notebook

This notebook produces **two** result rows from a single training run: retrieval
*without* the app-metadata half and retrieval *with* it. The cells are ordered so
that "Run All" works top to bottom and nothing is evaluated before it exists:

| step | cell | needs | produces |
|---|---|---|---|
| 1 | `9_train_joint.py` | `embeddings_4096_last/` | `EMB_DIR/`, `LSTM_PATH`, projector |
| 2 | `4_evaluate_screen_prediction.py` on `EMB_DIR` | step 1 | **result row: no app desc** |
| 3 | `6_generate_app_context_embeddings.py` | step 1 | `EMB_APP_DIR/` |
| 4 | `4_evaluate_screen_prediction.py` on `EMB_APP_DIR` | steps 1 + 3 | **result row: with app desc** |

The no-app evaluation runs before step 3 on purpose: it depends only on step 1,
while step 3 embeds every Play-Store description with SBERT and takes about an
hour. Nothing after step 2 touches `EMB_DIR`, so both rows come from the same
projector and the same LSTM checkpoint - they are a clean ablation of the app
half, not two separate runs.

### What the app half is, and why both rows are reported

`6_generate_app_context_embeddings.py` embeds each screen's Play-Store app
description with SBERT (`sentence-transformers/all-mpnet-base-v2`, 768-d) and
concatenates that vector onto the projected screen vector. It is a *per-app*
constant: every screen of the same app carries the identical second half.

The LSTM never consumes it. `9_train_joint.py` trains on the raw Qwen embeddings
only, and at evaluation time `4_evaluate_screen_prediction.py` slices the screen
part off (`ctx_raw[:, :lstm_dim]`) before the forward pass. The app half enters
*only* in the ranking step, where it is copied verbatim from the **target** onto
the prediction (`4_evaluate_screen_prediction.py:373-375`). Step 2 is that same
evaluation with the app half absent - no code change is needed, because when the
embedding dimension equals `lstm_dim` the concatenation branch does not fire.

Why it matters: with the app half attached, every candidate
from the target's app scores high, and since a RICO app has ~7.1 screens on
average (median 5), the Top-0.01% ... Top-5% columns largely measure app identity
rather than screen prediction - a measured app-only oracle already reaches 0.9968
at Top-0.1% and 1.0 at Top-1%. In the step-2 row **every** column is a
screen-level number; in the step-4 row only Top-1 and MRR are.

**Artefacts are suffixed `_strict`** and cannot clobber a permissive or an
n-back run:

    embeddings_seq_768_strict/                    (9_)
    lstm_predictor_768d_strict.pth                (9_)
    sequence_projector_768d_strict.pth            (9_)
    embeddings_seq_768_strict_with_app_desc/      (6_)

`6_generate_app_context_embeddings.py` needs no `--strict` of its own - it just
concatenates an app vector per screen, so it is pointed at the strict
directories instead.

### Caveats on the numbers this produces

1. **Screen ordering.** Screen2Vec builds each trace from an unsorted
   `os.listdir()` of `view_hierarchies`
   (`UI_embedding/precompute_embeddings.py:72`), which is not temporal order and
   is not reproducible on another filesystem. This pipeline always uses
   `gestures.json` order, i.e. true temporal order.

2. **Only the step-4 row is protocol-comparable to the paper.**
   Screen2Vec's Table 3 numbers were produced *with* the concatenated
   app-description half. Read the step-2 row against the step-4 row to size that
   effect; do not put a no-app Top-1% next to a paper Top-1% and call it a
   reproduction.

3. **Trace and layout id namespaces are joined through the index.**
   Embeddings are keyed by Rico's *global* `data/layouts/` numbering while
   `gestures.json` keys are per-trace indices; every stage here resolves them
   through `trace_layout_index.csv` (`0_build_trace_index.py`) and refuses to run
   without it. Unmatched trace screens are dropped, so the evaluated population
   is the matched subset.

In [1]:
# ==== Configuration: strict protocol ====
PROJ_DIM = 768   # projector output dimension

EMB_DIR     = f"embeddings_seq_{PROJ_DIM}_strict"        # 9_ output (step 1)
EMB_APP_DIR = f"{EMB_DIR}_with_app_desc"                 # 6_ output (step 3)
LSTM_PATH   = f"lstm_predictor_{PROJ_DIM}d_strict.pth"   # 9_ output (step 1)

# Result-row labels. Kept inside the 45-char label column of
# print_comparison_table() so the two rows stay aligned with the paper's.
LABEL_NOAPP = ">>> Qwen3-VL+LSTM (no app) [strict] <<<"
LABEL_APP   = ">>> Qwen3-VL+LSTM (with app) [strict] <<<"

print(f"protocol = strict  |  proj_dim = {PROJ_DIM}")
print(f"  projected embeddings : {EMB_DIR}/")
print(f"  + app context        : {EMB_APP_DIR}/")
print(f"  LSTM checkpoint      : {LSTM_PATH}")

protocol = strict  |  proj_dim = 768
  projected embeddings : embeddings_seq_768_strict/
  + app context        : embeddings_seq_768_strict_with_app_desc/
  LSTM checkpoint      : lstm_predictor_768d_strict.pth


In [2]:
# Step 1/4 - train the projector and the LSTM jointly on the raw 4096-d Qwen
# embeddings. Writes EMB_DIR/, LSTM_PATH and the projector checkpoint. Both
# evaluations below read these, so this cell must run first.
%run 9_train_joint.py --proj_dim $PROJ_DIM --strict

Device: cpu
Joint training: Projector (4096 → 1024 → 768) + LSTM (768)
Loading 66261 embeddings from 'embeddings_4096_last' in parallel...


Loading embeddings: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 66261/66261 [00:50<00:00, 1306.47it/s]


  Loaded matrix shape: (66261, 4096)
  Loaded trace->layout index: 66157 screens ('trace_layout_index.csv')
Parsing 10292 traces in parallel...


Parsing traces: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 10292/10292 [00:03<00:00, 2722.54it/s]


  Parsed 5095 traces
  STRICT: traces >= 5 screens, negative pool = 54520 of 66261 screens
  Input dim: 4096  |  Traces: 5095  |  Screens: 66261
  Train traces: 4586  |  Val traces: 509
  Projector parameters: 4,980,736
  LSTM parameters:      4,724,736
  Total parameters:     9,705,472

  Joint Training: 120 epochs, batch_size=256
  neg_samp=128, n_predictors=4
  temperature=0.07
  trace_negatives=context

Epoch   1/120 | Train: 5.2249 | Val: 3.5561 * best
Epoch   2/120 | Train: 3.3704 | Val: 2.9846 * best
Epoch   3/120 | Train: 2.8740 | Val: 2.7288 * best
Epoch   4/120 | Train: 2.6621 | Val: 2.6495 * best
Epoch   5/120 | Train: 2.5416 | Val: 2.6974 (patience 1/15)
Epoch   6/120 | Train: 2.4106 | Val: 2.6658 (patience 2/15)
Epoch   7/120 | Train: 2.3006 | Val: 2.6101 * best
Epoch   8/120 | Train: 2.2462 | Val: 2.5641 * best
Epoch   9/120 | Train: 2.1553 | Val: 2.6166 (patience 1/15)
Epoch  10/120 | Train: 2.1003 | Val: 2.5168 * best
Epoch  11/120 | Train: 2.0711 | Val: 2.5587 (patienc

Projecting: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 17/17 [00:03<00:00,  4.52it/s]


  Projected shape: (66261, 768)


Saving embeddings: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 66261/66261 [02:00<00:00, 551.53it/s]


Saved LSTM predictor: lstm_predictor_768d_strict.pth
Saved projector: sequence_projector_768d_strict.pth
Saved training plot: joint_training_768d_strict.png

Total training time: 3674.2s

Next steps:
  python 6_generate_app_context_embeddings.py \
    --embeddings_dir embeddings_seq_768_strict \
    --output_dir embeddings_seq_768_strict_with_app

  python 4_evaluate_screen_prediction.py \
    --embeddings_dir embeddings_seq_768_strict_with_app \
    --lstm_model lstm_predictor_768d_strict.pth \
    --strict


In [3]:
# Step 2/4 - evaluate WITHOUT the app-metadata half: retrieval over the plain
# PROJ_DIM-d screen vectors in EMB_DIR. Depends only on step 1, so it runs before
# the expensive SBERT stage. Every column of this row is a screen-level number.
%run 4_evaluate_screen_prediction.py --embeddings_dir $EMB_DIR --lstm_model $LSTM_PATH --strict --label "$LABEL_NOAPP"


[Step 0/3] Loading LSTM predictor from 'lstm_predictor_768d_strict.pth'...
  Loaded LSTM predictor (embed_dim=768, best_val_loss=2.5168)

[Step 1/3] Loading embeddings...
Loading 66261 embeddings from 'embeddings_seq_768_strict' in parallel...


Loading embeddings: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 66261/66261 [00:51<00:00, 1276.93it/s]


  Loaded matrix shape: (66261, 768)
  66261 screen embeddings available

[Step 2/3] Parsing RICO interaction traces...
  Loaded trace->layout index: 66157 screens ('trace_layout_index.csv')


Parsing traces: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 9384/9384 [00:10<00:00, 895.49it/s]


  Parsed 5095 traces (54520 screens)
  Skipped 5197 traces with < 5 screens
  Skipped 104 screens absent from the trace->layout index
  Strict pool: restricted to 54520 screens from eligible traces (was 66261)

[Step 3/3] Running screen prediction evaluation...

  Protocol: STRICT (Screen2Vec modeltester.py population, exactly 4 context screens per prediction)

  Total screens in embedding index: 54520
  Top-0.01% threshold: k=5
  Top-0.1%  threshold: k=54
  Top-1%    threshold: k=545
  Top-5%    threshold: k=2726
  Mean embedding L2 norm: 1.0000
  Prediction mode: LSTM (Screen2Vec-equivalent)
  Evaluating traces in parallel via threads (n_jobs=-1)...


Evaluating traces (LSTM): 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 5095/5095 [40:05<00:00,  2.12it/s]



  Total prediction instances: 34140

  GUI Screen Prediction — Comparison with Paper Table 3
Model                                            Top-1  Top-0.01%   Top-0.1%   Top-1%   Top-5%  Norm.RMSE
---------------------------------------------------------------------------------------------------------
>>> Qwen3-VL+LSTM (no app) [strict] <<<         0.0697     0.2443     0.6532   0.9159   0.9782     0.8525
---------------------------------------------------------------------------------------------------------
Screen2Vec-EUCLIDEAN-spatial                    0.0610     0.2580     0.9690   0.9980   1.0000     0.8530
Screen2Vec-EUCLIDEAN-no spatial                 0.0065     0.1160     0.8960   0.9860   0.9990     1.7230
TextOnly                                        0.0120     0.0550     0.1960   0.4390   0.6430     1.2410
LayoutOnly                                      0.0041     0.0240     0.0910   0.2220   0.3950     1.1350
VisualOnly                                      0.0060    

In [4]:
# Step 3/4 - concatenate the 768-d SBERT app-description vector onto every screen
# vector. Reads EMB_DIR (which step 2 left untouched) and writes EMB_APP_DIR.
# Slow: it embeds every Play-Store description, roughly an hour.
%run 6_generate_app_context_embeddings.py --embeddings_dir $EMB_DIR --output_dir $EMB_APP_DIR

C:\Users\Jonathan\Dev\screen2vec-revisited\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading app details from ../rico-data/data/play_store_metadata/app_details_enriched.csv...
  CSV columns: ['App Package Name', 'Play Store Name', 'Category', 'Average Rating', 'Number of Ratings', 'Number of Downloads', 'Date Updated', 'Icon URL', 'gp_title', 'gp_description', 'gp_descriptionHTML', 'gp_summary', 'gp_installs', 'gp_minInstalls', 'gp_realInstalls', 'gp_score', 'gp_ratings', 'gp_reviews', 'gp_histogram', 'gp_price', 'gp_free', 'gp_currency', 'gp_sale', 'gp_saleTime', 'gp_originalPrice', 'gp_saleText', 'gp_offersIAP', 'gp_inAppProductPrice', 'gp_developer', 'gp_developerId', 'gp_developerEmail', 'gp_developerWebsite', 'gp_developerAddress', 'gp_privacyPolicy', 'gp_genre', 'gp_genreId', 'gp_categories', 'gp_icon', 'gp_headerImage', 'gp_screenshots', 'gp_video', 'gp_videoImage', 'gp_contentRating', 'gp_contentRatingDescription', 'gp_adSupported', 'gp_containsAds', 'gp_released', 'gp_lastUpdatedOn', 'gp_updated', 'gp_version', 'gp_comments', 'gp_status']
  Found description c

Parsing layout JSONs: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 104/104 [00:00<00:00, 47413.87it/s]


Preparing text input for embedding model...
Loading SBERT model 'sentence-transformers/all-mpnet-base-v2' (embed_dim=768)...


Loading weights: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 199/199 [00:00<00:00, 2830.47it/s]
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     | Details
------------------------+------------+--------
embeddings.position_ids | UNEXPECTED |        

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding app packages...


... (2 outputs left out)

Saving concatenated embeddings:   0%|                                                                                                                                       | 47/66261 [00:00<04:42, 234.46it/s]

Concatenating screen (768-dim) + app (768-dim) embeddings...
Output dimension: 1536-dim


Saving concatenated embeddings: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 66261/66261 [04:08<00:00, 266.66it/s]


Success! Concatenated embeddings created and saved.


In [5]:
# Step 4/4 - evaluate WITH the app-metadata half: same LSTM checkpoint as step 2,
# retrieval over the PROJ_DIM+768-d vectors from step 3. This is the
# paper-comparable row; see the caveats at the top on how to read its columns.
%run 4_evaluate_screen_prediction.py --embeddings_dir $EMB_APP_DIR --lstm_model $LSTM_PATH --strict --label "$LABEL_APP"


[Step 0/3] Loading LSTM predictor from 'lstm_predictor_768d_strict.pth'...
  Loaded LSTM predictor (embed_dim=768, best_val_loss=2.5168)

[Step 1/3] Loading embeddings...
Loading 66261 embeddings from 'embeddings_seq_768_strict_with_app_desc' in parallel...


Loading embeddings: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 66261/66261 [00:53<00:00, 1229.63it/s]


  Loaded matrix shape: (66261, 1536)
  66261 screen embeddings available

[Step 2/3] Parsing RICO interaction traces...
  Loaded trace->layout index: 66157 screens ('trace_layout_index.csv')


Parsing traces: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 9384/9384 [00:12<00:00, 751.22it/s]


  Parsed 5095 traces (54520 screens)
  Skipped 5197 traces with < 5 screens
  Skipped 104 screens absent from the trace->layout index
  Strict pool: restricted to 54520 screens from eligible traces (was 66261)

[Step 3/3] Running screen prediction evaluation...

  Protocol: STRICT (Screen2Vec modeltester.py population, exactly 4 context screens per prediction)

  Total screens in embedding index: 54520
  Top-0.01% threshold: k=5
  Top-0.1%  threshold: k=54
  Top-1%    threshold: k=545
  Top-5%    threshold: k=2726
  Mean embedding L2 norm: 1.4142
  Prediction mode: LSTM (Screen2Vec-equivalent)
  Evaluating traces in parallel via threads (n_jobs=-1)...


Evaluating traces (LSTM): 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 5095/5095 [1:02:57<00:00,  1.35it/s]



  Total prediction instances: 34140

  GUI Screen Prediction — Comparison with Paper Table 3
Model                                            Top-1  Top-0.01%   Top-0.1%   Top-1%   Top-5%  Norm.RMSE
---------------------------------------------------------------------------------------------------------
>>> Qwen3-VL+LSTM (with app) [strict] <<<       0.1182     0.5004     0.9917   0.9994   1.0000     0.6028
---------------------------------------------------------------------------------------------------------
Screen2Vec-EUCLIDEAN-spatial                    0.0610     0.2580     0.9690   0.9980   1.0000     0.8530
Screen2Vec-EUCLIDEAN-no spatial                 0.0065     0.1160     0.8960   0.9860   0.9990     1.7230
TextOnly                                        0.0120     0.0550     0.1960   0.4390   0.6430     1.2410
LayoutOnly                                      0.0041     0.0240     0.0910   0.2220   0.3950     1.1350
VisualOnly                                      0.0060    